In [1]:
import sys
import os

if not os.path.exists("config.py"):
    os.chdir("backend") if os.path.exists("backend") else os.chdir("..")

sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend
config.py exists: True


In [2]:
import io
import re
import pdfplumber
from openai import OpenAI
from supabase import create_client
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from googleapiclient.http import MediaIoBaseDownload
from config import settings

openai_client = OpenAI(api_key=settings.openai_api_key)
sb = create_client(settings.supabase_url, settings.supabase_service_role_key)

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
creds = Credentials.from_authorized_user_file(settings.google_token_file, SCOPES)
service = build("drive", "v3", credentials=creds)

print("Clients ready")

Clients ready


In [4]:
PDF_FILE_ID = "1c-18lHMiW-wxFnUlhd6y21RNlP6ex_Ek"

def download_file(file_id: str) -> bytes:
    request = service.files().get_media(fileId=file_id)
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return buffer.getvalue()

def clean_text(text: str) -> str:
    text = re.sub(r'\.{4,}\s*\d+', '', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def chunk_by_clauses(text: str, max_tokens: int = 1500) -> list[dict]:
    text = clean_text(text)
    pattern = re.compile(r'(?m)^(\d+\.\d+(?:\.\d+)?)\s+(.+)')
    matches = list(pattern.finditer(text))
    chunks = []
    for i, match in enumerate(matches):
        clause_ref = match.group(1)
        clause_title = match.group(2).strip()
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end].strip()
        if len(content) < 50:
            continue
        words = content.split()
        if len(words) > max_tokens:
            step = max_tokens
            overlap = 100
            for j in range(0, len(words), step - overlap):
                sub_content = " ".join(words[j:j + step])
                chunks.append({
                    "clause_ref": f"{clause_ref} (part {j // (step - overlap) + 1})",
                    "clause_title": clause_title,
                    "content": sub_content,
                    "char_start": start,
                    "char_end": end,
                })
        else:
            chunks.append({
                "clause_ref": clause_ref,
                "clause_title": clause_title,
                "content": content,
                "char_start": start,
                "char_end": end,
            })
    return chunks

pdf_bytes = download_file(PDF_FILE_ID)

with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
    full_text = "\n".join(
        page.extract_text() for page in pdf.pages[8:]
        if page.extract_text()
    )

chunks = chunk_by_clauses(full_text)
print(f"Chunks ready: {len(chunks)}")

Chunks ready: 773


In [8]:
def embed_text(text: str) -> list[float]:
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# Test on first chunk
test_chunk = chunks[0]
embedding = embed_text(test_chunk["content"])

print(f"Chunk: {test_chunk['clause_ref']}")
print(f"Embedding length: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}